In [ ]:
!pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib
!pip install beautifulsoup4 html2text  # for email content cleaning
!pip install langchain-community

In [ ]:
import os
import pickle
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

# If modifying these scopes, delete the file token.pickle.
SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

class GmailConnector:
    def __init__(self, credentials_file='../credentials.json'):
        self.credentials_file = credentials_file
        self.service = None
        self.authenticate()
    
    def authenticate(self):
        """Authenticate and create Gmail service"""
        creds = None
        
        # The file token.pickle stores the user's access and refresh tokens.
        if os.path.exists('token.pickle'):
            with open('token.pickle', 'rb') as token:
                creds = pickle.load(token)
        
        # If there are no (valid) credentials available, let the user log in.
        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                creds.refresh(Request())
            else:
                flow = InstalledAppFlow.from_client_secrets_file(
                    self.credentials_file, SCOPES)
                creds = flow.run_local_server(port=0)
            
            # Save the credentials for the next run
            with open('token.pickle', 'wb') as token:
                pickle.dump(creds, token)
        
        self.service = build('gmail', 'v1', credentials=creds)
        print("Successfully connected to Gmail!")
    
    def get_messages(self, query='', max_results=10):
        """Get messages based on query"""
        try:
            # Get message IDs
            result = self.service.users().messages().list(
                userId='me', 
                q=query, 
                maxResults=max_results
            ).execute()
            
            messages = result.get('messages', [])
            return messages
        except Exception as error:
            print(f'An error occurred: {error}')
            return []
    
    def get_message_content(self, message_id):
        """Get full message content"""
        try:
            message = self.service.users().messages().get(
                userId='me', 
                id=message_id, 
                format='full'
            ).execute()
            return message
        except Exception as error:
            print(f'An error occurred: {error}')
            return None

# Test the connection
gmail = GmailConnector()
messages = gmail.get_messages("in:inbox", max_results=20)
print(f"Found {len(messages)} messages")

In [ ]:
import base64
import re
from bs4 import BeautifulSoup
import html2text
from datetime import datetime

class EmailProcessor:
    def __init__(self):
        self.html_converter = html2text.HTML2Text()
        self.html_converter.ignore_links = True
        self.html_converter.ignore_images = True
    
    def extract_email_data(self, message):
        """Extract structured data from Gmail message"""
        payload = message['payload']
        headers = payload.get('headers', [])
        
        # Extract headers
        email_data = {
            'id': message['id'],
            'thread_id': message['threadId'],
            'subject': '',
            'sender': '',
            'date': '',
            'body': '',
            'labels': message.get('labelIds', [])
        }
        
        # Parse headers
        for header in headers:
            name = header['name'].lower()
            if name == 'subject':
                email_data['subject'] = header['value']
            elif name == 'from':
                email_data['sender'] = header['value']
            elif name == 'date':
                email_data['date'] = header['value']
        
        # Extract body
        email_data['body'] = self._extract_body(payload)
        
        return email_data
    
    def _extract_body(self, payload):
        """Extract body text from email payload"""
        body = ""
        
        if 'parts' in payload:
            # Multi-part message
            for part in payload['parts']:
                if part['mimeType'] == 'text/plain':
                    if 'data' in part['body']:
                        body = base64.urlsafe_b64decode(
                            part['body']['data']
                        ).decode('utf-8')
                        break
                elif part['mimeType'] == 'text/html':
                    if 'data' in part['body']:
                        html_body = base64.urlsafe_b64decode(
                            part['body']['data']
                        ).decode('utf-8')
                        body = self.html_converter.handle(html_body)
        else:
            # Single-part message
            if payload['mimeType'] == 'text/plain':
                if 'data' in payload['body']:
                    body = base64.urlsafe_b64decode(
                        payload['body']['data']
                    ).decode('utf-8')
            elif payload['mimeType'] == 'text/html':
                if 'data' in payload['body']:
                    html_body = base64.urlsafe_b64decode(
                        payload['body']['data']
                    ).decode('utf-8')
                    body = self.html_converter.handle(html_body)
        
        return self._clean_text(body)
    
    def _clean_text(self, text):
        """Clean and normalize text"""
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text)
        # Remove email signatures (basic)
        text = re.sub(r'\n--\n.*', '', text, flags=re.DOTALL)
        return text.strip()

# Test the processor
processor = EmailProcessor()

for msg in messages[:3]:
    full_message = gmail.get_message_content(msg['id'])
    if full_message:
        email_data = processor.extract_email_data(full_message)
        print(f"\nSubject: {email_data['subject']}")
        print(f"From: {email_data['sender']}")
        print(f"Body preview: {email_data['body'][:200]}...")

In [45]:
from dotenv import load_dotenv
from langchain.schema import Document
from langchain.text_splitter import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

In [ ]:
# Load environment variables
load_dotenv(override=True)

db_name = "gmail_vector_db"
embeddings = OpenAIEmbeddings()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

def process_emails_to_documents(query="in:anywhere -in:trash -in:spam", max_results=150):
    """Convert Gmail emails to LangChain documents"""
    print(f"Fetching emails with query: {query}")
    messages = gmail.get_messages(query, max_results)
    
    documents = []
    for i, msg in enumerate(messages):
        print(f"Processing email {i+1}/{len(messages)}")
        
        full_message = gmail.get_message_content(msg['id'])
        if full_message:
            email_data = processor.extract_email_data(full_message)
            
            # Create document content
            content = f"Subject: {email_data['subject']}\n"
            content += f"From: {email_data['sender']}\n"
            content += f"Date: {email_data['date']}\n\n"
            content += email_data['body']
            
            # Convert labels list to string for Chroma compatibility
            labels_str = ", ".join(email_data['labels']) if email_data['labels'] else ""
            
            # Create LangChain document
            doc = Document(
                page_content=content,
                metadata={
                    'doc_type': 'email',
                    'email_id': str(email_data['id']),
                    'subject': str(email_data['subject'])[:500],
                    'sender': str(email_data['sender'])[:200],
                    'date': str(email_data['date']),
                    'labels': labels_str
                }
            )
            documents.append(doc)
    
    print(f"Documents {documents}")
    return documents
  
process_emails_to_documents()

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(process_emails_to_documents())
print(f"Total chunks: {len(chunks)}")


In [ ]:
# Create vector store
embeddings = OpenAIEmbeddings()

if os.path.exists(db_name):
    print(f"🔵 Deleting existing database")
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()
    
print(f"🔵 About to call Chroma.from_documents...")
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"🔵 SUCCESS: Vectorstore created with {vectorstore._collection.count()} documents")


In [ ]:
# Investigate the vector store
collection = vectorstore._collection
count = collection.count()

sample_embeddings = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embeddings)
print(f"There are {count:,} vectors with {dimensions} dimensions in the vector store")

In [33]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np
import plotly.graph_objects as go

In [ ]:
# Visualizing the vector store
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
subjects = [item['subject'] for item in result['metadatas']]
subjects_short = [subject[:50] if len(subject) > 50 else subject for subject in subjects]

# Filter to first occurrence of each unique subject
seen_subjects = {}
unique_indices = []
unique_subjects = []
unique_vectors = []

for i, subject in enumerate(subjects):
    if subject not in seen_subjects:
        seen_subjects[subject] = True
        unique_indices.append(i)
        unique_subjects.append(subject)
        unique_vectors.append(vectors[i])

# Create categories based on UNIQUE subjects
unique_subjects_short = [subject[:50] if len(subject) > 50 else subject for subject in unique_subjects]

def categorize_subject(subject):
    subject_lower = subject.lower()
    if any(word in subject_lower for word in ['job', 'engineer', 'linkedin', 'career']):
        return 'Jobs'
    elif any(word in subject_lower for word in ['newsletter', 'dev', 'bytes', 'pragmatic', 'smashing']):
        return 'Tech News'
    elif any(word in subject_lower for word in ['american express', 'transaction', 'card']):
        return 'Banking'
    elif any(word in subject_lower for word in ['learn', 'course', 'freecodecamp']):
        return 'Learning'
    elif any(word in subject_lower for word in ['booking', 'airbnb', 'viaje', 'travel', 'trip', 'passport', 'airline', 'pasaporte', 'vacation']):
        return 'Travel'
    else:
        return 'Other'

# Create categories for unique subjects only
unique_categories = [categorize_subject(subject) for subject in unique_subjects]
# categories = [categorize_subject(subject) for subject in subjects] # Original categories
color_map = {'Jobs': 'red', 'Tech News': 'blue', 'Banking': 'green', 'Learning': 'orange', 'Travel': 'pink', 'Other': 'gray'}
colors = [color_map[cat] for cat in unique_categories]

# Use unique_vectors and unique_subjects for visualization
tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, len(unique_vectors)-1))
reduced_vectors = tsne.fit_transform(np.array(unique_vectors))

# Create a 2D scatter plot of the vectors
fig = go.Figure()

for category in color_map.keys():
    # Filter data for this category
    category_indices = [i for i, cat in enumerate(unique_categories) if cat == category]
    if category_indices:  # Only add if category has data
        fig.add_trace(go.Scatter(
            x=[reduced_vectors[i, 0] for i in category_indices],
            y=[reduced_vectors[i, 1] for i in category_indices],
            mode='markers',
            marker=dict(
                color=color_map[category],
                size=10,
                opacity=0.8
            ),
            text=[unique_subjects_short[i] for i in category_indices],
            name=category,
            hovertemplate='<b>%{text}</b><br>Category: ' + category + '<extra></extra>'
        ))

fig.update_layout(
    title='Email Vector Store Visualization by Subject Type',
    xaxis_title='t-SNE Component 1',
    yaxis_title='t-SNE Component 2',
    width=800,
    height=600,
    margin=dict(l=50, r=50, b=50, t=50)
)

fig.show()

In [ ]:
# Let's try in 3D
tsne = TSNE(n_components=3, random_state=42, perplexity=min(5, len(unique_vectors)-1))
reduced_vectors = tsne.fit_transform(np.array(unique_vectors))

# Create a 3D scatter plot of the vectors
fig = go.Figure()

for category in color_map.keys():
    # Filter data for this category
    category_indices = [i for i, cat in enumerate(unique_categories) if cat == category]
    if category_indices:  # Only add if category has data
        fig.add_trace(go.Scatter3d(
            x=[reduced_vectors[i, 0] for i in category_indices],
            y=[reduced_vectors[i, 1] for i in category_indices],
            z=[reduced_vectors[i, 2] for i in category_indices],
            mode='markers',
            marker=dict(
                color=color_map[category],
                size=10,
                opacity=0.8
            ),
            text=[unique_subjects_short[i] for i in category_indices],
            name=category,
            hovertemplate='<b>%{text}</b><br>Category: ' + category + '<extra></extra>'
        ))

fig.update_layout(
    title='Email Vector Store Visualization by Subject Type in 3D',
    scene=dict(
        xaxis_title='t-SNE Component 1',
        yaxis_title='t-SNE Component 2',
        zaxis_title='t-SNE Component 3'
    ),
    width=900,
    height=900,
    margin=dict(l=50, r=50, b=50, t=50)
)

fig.show()
